# USGS: observaciones instrumentales y macroseísmicas de Cali

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/usgs_estaciones_cali.geojson`. Una fila representa un registro de stationlist de ShakeMap; puede ser estación instrumental o celda macroseísmica DYFI.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/usgs_estaciones_cali.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '16_usgs_estaciones_cali', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Naturaleza de las observaciones y faltantes

In [ ]:
for c in ["station_type","network","source"]:display(conteos(df[c],c))
display(faltantes(df,["code","pga","pgv","intensity","distance","vs30"]))
tipos=conteos(df.station_type,"tipo_observacion")
grafico_barras(tipos,"Separar instrumentos de observaciones macroseísmicas", "16_tipos")
resumen.update({"tipos_estacion":df.station_type.value_counts().to_dict(),
 "codigo_duplicados":int(df.duplicated(["network","code"]).sum()),
 "pga_con_dato":int(df.pga.notna().sum()),"intensidad_con_dato":int(df.intensity.notna().sum())})

## Reglas decididas para plata

- **Seleccionar como evidencia de observaciones que sustentan ShakeMap**, en tablas separadas por tipo y con unidad/versión explícita.
- No describir todas las filas como acelerógrafos: DYFI resume percepción ciudadana. Conservar intensidad macroseísmica y movimiento instrumental como medidas distintas.
- Auditar procedencia y si cada observación fue insumo de la versión del modelo evaluada. Esa comparación mide consistencia interna; una validación independiente exige observaciones retenidas u otra evidencia.
- No usar un único instrumento para inferir cobertura instrumental por comuna ni capacidad de respuesta.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('16_usgs_estaciones_cali', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")